# P3 DPP — RepDWNet + WDice on Kaggle

Upload this notebook, attach PhenoBench, enable Internet and select **GPU T4 ×2**. Edit the first config cell, then Save & Run All. This notebook clones source from GitHub; push `f/down-dpp` before running. One run produces one artifact ZIP.

Only `lite_dpp_sym` is implemented. Future variants must be implemented in source before they can be selected here. Existing P2 configs and training recipe are retained: 100 epochs, validation every 5 epochs, WDice, runtime 2 devices.

DPP initialization follows the authors’ released code: log-alpha/log-lambda are independently sampled from Uniform(-1/sqrt(C), +1/sqrt(C)), centered at zero for the default alpha_init=lambda_init=1. The paper describes Gaussian perturbation; this implementation follows the published code instead. No manually selected noise standard deviation is used.

Forward follows `visinf_package/DPP_sym_lite.lua` at author commit `e810bd63ed61bd7710559467bcd9a35ab9a1b028`: normalize squared deviation plus epsilon by its local mean before the reward exponent; cap log-alpha at 10 as in `PositiveBias.lua`. This is the graph implementation, not the separate CUDA implementation.


In [ ]:
# FIRST CONFIG CELL: choose the experiment and source here.
REPO_URL = 'https://github.com/FUAVX/RPD_Reproduce.git'
GIT_REF = 'f/down-dpp'
SOURCE_COMMIT = ''  # optional full commit SHA reachable from GIT_REF
EXPERIMENT_MODE = 'ex1'  # ex1 = Early, ex2 = Late, ex3 = All
DPP_VARIANT = 'lite_dpp_sym'  # future: lite_dpp_asym/full_dpp; currently unsupported
RUN_TAG = 'run1'  # use a new tag for a separate repeat
DATASET_DIR = ''  # empty = auto-detect attached PhenoBench Kaggle input
NUM_WORKERS = 2
DEVICES = 2  # P2 runtime override; YAML default remains 1
WORKING_ROOT = '/kaggle/working'

EXPERIMENTS = {
    'ex1': ('P3_DPP_Ex1_Early', [1, 2]),
    'ex2': ('P3_DPP_Ex2_Late', [3, 4]),
    'ex3': ('P3_DPP_Ex3_All', [1, 2, 3, 4]),
}
if EXPERIMENT_MODE not in EXPERIMENTS:
    raise ValueError(f'Unknown EXPERIMENT_MODE: {EXPERIMENT_MODE}')
if DPP_VARIANT != 'lite_dpp_sym':
    raise NotImplementedError(f'{DPP_VARIANT} is not implemented. Implement and test the variant before selecting it.')
if DEVICES != 2:
    raise ValueError('This P2-matched protocol requires 2 devices.')
SCENARIO, EXPECTED_STAGES = EXPERIMENTS[EXPERIMENT_MODE]


In [ ]:
# Kaggle runtime dependencies. Kaggle supplies torch/CUDA; do not reinstall torch.
!nvidia-smi
# Lightning 1.9.5 uses np.Inf; keep NumPy 1.x as specified in P2 requirements.
!pip install -q 'numpy<2' 'pytorch-lightning==1.9.5' 'torchmetrics==0.11.4' 'setuptools<70' tensorboard matplotlib pandas tabulate timm scikit-image tifffile albumentations pyyaml rich


In [ ]:
import json
import pathlib
import shutil
import subprocess
import yaml

WORKING_DIR = pathlib.Path(WORKING_ROOT)
REPO_DIR = WORKING_DIR / 'RPD_P2'
RUN_DIR = WORKING_DIR / f'{SCENARIO}_{DPP_VARIANT}_{RUN_TAG}'
if RUN_DIR.exists():
    raise FileExistsError(f'Output already exists: {RUN_DIR}. Choose a new RUN_TAG.')

def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(['git', 'clone', '--branch', GIT_REF, '--single-branch', REPO_URL, str(REPO_DIR)])
if SOURCE_COMMIT:
    run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR)
run(['git', 'rev-parse', '--verify', 'HEAD'], cwd=REPO_DIR)

scenario_path = REPO_DIR / 'configs' / 'p3_scenarios' / f'{SCENARIO}.yaml'
required_paths = [scenario_path, REPO_DIR / 'test_wdice.py', REPO_DIR / 'test_dpp.py',
                  REPO_DIR / 'models/rpdnet/dpp.py', REPO_DIR / 'callbacks/dpp_logging.py']
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Branch does not contain the committed P3 DPP files: ' + ', '.join(missing))

def is_phenobench_root(path):
    return (
        (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'semantics').is_dir()
        and (path / 'val' / 'images').is_dir()
        and (path / 'val' / 'semantics').is_dir()
    )

if DATASET_DIR:
    dataset_dir = pathlib.Path(DATASET_DIR)
else:
    kaggle_input = pathlib.Path('/kaggle/input')
    candidates = [
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset',
        kaggle_input / 'phenobench-dataset',
    ]
    if kaggle_input.is_dir():
        attached_inputs = [p for p in kaggle_input.iterdir() if p.is_dir()]
        candidates.extend(p / 'PhenoBench' for p in attached_inputs)
        candidates.extend(attached_inputs)
    dataset_dir = next((p for p in candidates if is_phenobench_root(p)), None)
if dataset_dir is None or not is_phenobench_root(dataset_dir):
    raise FileNotFoundError('Cannot find a complete PhenoBench root in /kaggle/input. Attach the PhenoBench dataset, or set DATASET_DIR to its root containing train/images, train/semantics, val/images, and val/semantics.')

print(f'[OK] source ref: {subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()}')
print(f'[OK] dataset: {dataset_dir}')
print(f'[OK] train images: {len(list((dataset_dir / "train" / "images").glob("*")))}')


In [ ]:
# Resolve the Kaggle-specific configuration without modifying the cloned repository.
RUN_DIR.mkdir(parents=True, exist_ok=True)
with scenario_path.open(encoding='utf-8') as stream:
    cfg = yaml.safe_load(stream)

cfg['data']['path_to_dataset'] = str(dataset_dir)
cfg['data']['num_workers'] = NUM_WORKERS
active_cfg_path = RUN_DIR / f'active_config_{SCENARIO}.yaml'
with active_cfg_path.open('w', encoding='utf-8') as stream:
    yaml.safe_dump(cfg, stream, sort_keys=False)

assert cfg['train']['loss'] == 'weighted_dice'
assert cfg['loss']['dice']['lambda'] == 1.0
assert cfg['loss']['dice']['epsilon'] == 1e-6
assert cfg['train']['class_weights'] == [1.47, 5.06, 10.02]
assert cfg['train']['max_epoch'] == 100
assert cfg['val']['check_val_every_n_epoch'] == 5
assert cfg['train']['batch_size'] == 4
assert cfg['train']['early_stopping']['mode'] == 'none'
assert cfg['backbone']['downsampling']['variant'] == DPP_VARIANT
assert cfg['backbone']['downsampling']['stages'] == EXPECTED_STAGES
# Match against the source P2 config to catch unintended recipe drift.
import copy
base_cfg = yaml.safe_load((REPO_DIR / 'configs/p2_scenarios/P2_Ex7_RepDWNet_WDice-Loss.yaml').read_text())
base_cfg['data']['path_to_dataset'] = str(dataset_dir)
base_cfg['data']['num_workers'] = NUM_WORKERS
inherited = copy.deepcopy(cfg)
del inherited['backbone']['downsampling']
inherited['experiment'] = base_cfg['experiment']
assert inherited == base_cfg, 'P3 differs from P2 beyond experiment metadata/downsampling.'
print(yaml.safe_dump({
    'experiment': cfg['experiment'],
    'backbone': cfg['backbone'],
    'loss': cfg['loss'],
}, sort_keys=False))


In [ ]:
# Check hardware and both WDice/DPP before full DDP training.
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != DEVICES:
    raise RuntimeError('Select GPU T4 x2 in Kaggle settings before running.')
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(DEVICES)])
run(['python', 'test_wdice.py'], cwd=REPO_DIR)
run(['python', 'test_dpp.py'], cwd=REPO_DIR)

TRAIN_LOG_DIR = RUN_DIR / 'training_output'
train_command = [
    'python', 'multi_metric_train.py', '--config', str(active_cfg_path),
    '--export_dir', str(TRAIN_LOG_DIR), '--devices', str(DEVICES),
]
# Persist provenance before launch so interrupted runs retain source/settings.
import importlib.metadata
source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
manifest = {
    'experiment': SCENARIO, 'mode': EXPERIMENT_MODE, 'variant': DPP_VARIANT, 'run_tag': RUN_TAG,
    'source_repository': REPO_URL, 'requested_ref': GIT_REF, 'requested_commit': SOURCE_COMMIT,
    'source_commit': source_commit, 'dataset_dir_at_runtime': str(dataset_dir),
    'seed': cfg['seed'], 'resolved_config': cfg,
    'dpp_implementation': {
        'reference_repository': 'https://github.com/visinf/dpp',
        'reference_commit': 'e810bd63ed61bd7710559467bcd9a35ab9a1b028',
        'reference_file': 'visinf_package/DPP_sym_lite.lua (nngraph)',
        'reward_normalization': 's / local_mean(s), s = deviation**2 + epsilon_squared',
        'log_parameter_initialization': 'Uniform(-1/sqrt(C), +1/sqrt(C)) at defaults',
        'log_alpha_max': 10,
    },
    'runtime_overrides': {'train.devices': DEVICES, 'val.devices': 1},
    'train_command': train_command, 'yaml_default_devices': cfg['train']['devices'],
    'batch_size_per_process': cfg['train']['batch_size'],
    'global_batch_size': cfg['train']['batch_size'] * DEVICES * cfg['train']['accumulate_grad_batches'],
    'gpu_names': [torch.cuda.get_device_name(i) for i in range(DEVICES)],
    'cuda_version': torch.version.cuda,
    'versions': {name: importlib.metadata.version(name) for name in
                 ('torch', 'pytorch-lightning', 'torchmetrics', 'numpy', 'PyYAML')},
    'status': 'started',
}
manifest_path = RUN_DIR / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
# Keep a resolved runtime config in addition to the P2-style active YAML snapshot.
runtime_cfg = copy.deepcopy(cfg)
runtime_cfg['train']['devices'] = DEVICES
runtime_cfg['val']['devices'] = 1  # matches the existing CLI override behavior
(RUN_DIR / 'runtime_config.yaml').write_text(yaml.safe_dump(runtime_cfg, sort_keys=False), encoding='utf-8')
try:
    run(train_command, cwd=REPO_DIR)
except Exception:
    manifest['status'] = 'failed'
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    raise
manifest['status'] = 'completed'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')


In [ ]:
# One ZIP per selected experiment/run, including all checkpoint/log/DPP exports.
if manifest['status'] != 'completed':
    raise RuntimeError('Training did not complete.')
required_outputs = [TRAIN_LOG_DIR / 'checkpoints/last.ckpt',
                    TRAIN_LOG_DIR / 'dpp_parameters/history.jsonl',
                    TRAIN_LOG_DIR / 'dpp_parameters/final.json',
                    TRAIN_LOG_DIR / 'dpp_parameters/checkpoint_val_weed_iou.json',
                    TRAIN_LOG_DIR / 'dpp_parameters/checkpoint_val_mIoU.json',
                    TRAIN_LOG_DIR / 'dpp_parameters/training_cost.json']
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing artifacts: ' + ', '.join(missing))
archive = shutil.make_archive(str(WORKING_DIR / f'{SCENARIO}_{DPP_VARIANT}_{RUN_TAG}_artifacts'),
                              'zip', root_dir=RUN_DIR)
print(f'[DONE] Download this Kaggle output: {archive}')
print(f'[DONE] Source commit: {source_commit}')
